# Data Cleaning - Brazilian E-Commerce Dataset
This notebook loads and merges the Olist dataset, handles missing values, converts dates, and creates derived features.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
# Load CSV files from data/raw/
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

print('Orders shape:', orders.shape)
print('Customers shape:', customers.shape)
print('Order items shape:', order_items.shape)
print('Products shape:', products.shape)
print('Reviews shape:', reviews.shape)

Orders shape: (99441, 8)
Customers shape: (99441, 5)
Order items shape: (112650, 7)
Products shape: (32951, 9)
Reviews shape: (99224, 7)


In [3]:
# Merge all 5 files into one master dataframe
# Start with orders as base
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(order_items, on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(reviews, on='order_id', how='left')

print('Master dataframe shape:', df.shape)

Master dataframe shape: (114092, 32)


In [4]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

Missing values per column:
order_id                              0
customer_id                           0
order_status                          0
order_purchase_timestamp              0
order_approved_at                   162
order_delivered_carrier_date       1980
order_delivered_customer_date      3253
order_estimated_delivery_date         0
customer_unique_id                    0
customer_zip_code_prefix              0
customer_city                         0
customer_state                        0
order_item_id                       778
product_id                          778
seller_id                           778
shipping_limit_date                 778
price                               778
freight_value                       778
product_category_name              2390
product_name_lenght                2390
product_description_lenght         2390
product_photos_qty                 2390
product_weight_g                    796
product_length_cm                   796
product_heigh

In [5]:
# Fix missing values
# Fill numeric columns with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Fill categorical columns with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print('Missing values after fixing:')
print(df.isnull().sum().sum())

Missing values after fixing:
0


In [6]:
# Convert date columns to proper datetime format
date_columns = ['order_purchase_timestamp', 'order_approved_at', 
                'order_delivered_carrier_date', 'order_delivered_customer_date',
                'order_estimated_delivery_date', 'review_creation_date',
                'review_answer_timestamp']

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print('Date columns converted to datetime')

Date columns converted to datetime


In [7]:
# Create new columns
# 1. delivery_days: how many days order took to deliver
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# 2. is_late: 1 if delivered after estimated date, 0 if not
df['is_late'] = np.where(df['order_delivered_customer_date'] > df['order_estimated_delivery_date'], 1, 0)

# 3. revenue: price + freight_value
df['revenue'] = df['price'] + df['freight_value']

print('New columns created: delivery_days, is_late, revenue')

New columns created: delivery_days, is_late, revenue


In [8]:
# Save final clean file to data/cleaned/master_data.csv
df.to_csv('../data/cleaned/master_data.csv', index=False)
print('Clean data saved to data/cleaned/master_data.csv')

Clean data saved to data/cleaned/master_data.csv


In [14]:
# Print shape and first 5 rows
print('Final dataframe shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

Final dataframe shape: (114092, 35)

First 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_width_cm,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,delivery_days,is_late,revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,13.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,Recomendo,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,8,0,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,19.0,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,13,0,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,21.0,e73b67b67587f7644d5bd1a52deb1b01,5.0,Recomendo,Muito bom,2018-08-18,2018-08-22 19:07:58,9,0,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,20.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,Recomendo,O produto foi exatamente o que eu esperava e e...,2017-12-03,2017-12-05 19:21:58,13,0,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,15.0,e50934924e227544ba8246aeb3770dd4,5.0,Recomendo,Muito bom,2018-02-17,2018-02-18 13:02:51,2,0,28.62
